# 1. Configurações iniciais:

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()

data_path = r"C:\Users\danil\OneDrive\Área de Trabalho\projeto-olist\data"

con.execute(f"""
    CREATE OR REPLACE VIEW orders AS
    SELECT * FROM read_csv_auto('{data_path}/olist_orders_dataset.csv');
    CREATE OR REPLACE VIEW customers AS
    SELECT * FROM read_csv_auto('{data_path}/olist_customers_dataset.csv');
    CREATE OR REPLACE VIEW order_items AS
    SELECT * FROM read_csv_auto('{data_path}/olist_order_items_dataset.csv');
    CREATE OR REPLACE VIEW products AS
    SELECT * FROM read_csv_auto('{data_path}/olist_products_dataset.csv');
    CREATE OR REPLACE VIEW reviews AS
    SELECT * FROM read_csv_auto('{data_path}/olist_order_reviews_dataset.csv');
    CREATE OR REPLACE VIEW payments AS
    SELECT * FROM read_csv_auto('{data_path}/olist_order_payments_dataset.csv');
    CREATE OR REPLACE VIEW sellers AS
    SELECT * FROM read_csv_auto('{data_path}/olist_sellers_dataset.csv');
""")

print("Pronto!")

Pronto!


# 2. Receita total e ticktet médio:

In [2]:
con.execute("""
    SELECT
        COUNT(DISTINCT o.order_id)            AS total_pedidos,
        ROUND(SUM(i.price), 2)                AS receita_total,
        ROUND(AVG(i.price), 2)                AS ticket_medio
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
""").fetchdf()

,total_pedidos,receita_total,ticket_medio
0,96478,13221498.11,119.98


# 3. Receita por mês:        

In [3]:
con.execute("""
    SELECT
        STRFTIME(o.order_purchase_timestamp, '%Y-%m') AS mes,
        COUNT(DISTINCT o.order_id)                    AS total_pedidos,
        ROUND(SUM(i.price), 2)                        AS receita
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY mes
    ORDER BY mes
""").fetchdf()

,mes,total_pedidos,receita
0,2016-09,1,134.97
1,2016-10,265,40325.11
2,2016-12,1,10.90
3,2017-01,750,111798.36
4,2017-02,1653,234223.40
5,2017-03,2546,359198.85
6,2017-04,2303,340669.68
7,2017-05,3546,489338.25
8,2017-06,3135,421923.37
9,2017-07,3872,481604.52


# 4. TOP 10 categorias por receita:

In [4]:
con.execute("""
    SELECT
        p.product_category_name             AS categoria,
        COUNT(DISTINCT o.order_id)          AS total_pedidos,
        ROUND(SUM(i.price), 2)              AS receita,
        ROUND(AVG(i.price), 2)              AS ticket_medio
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    JOIN products p    ON i.product_id = p.product_id
    WHERE o.order_status = 'delivered'
      AND p.product_category_name IS NOT NULL
    GROUP BY categoria
    ORDER BY receita DESC
    LIMIT 10
""").fetchdf()

,categoria,total_pedidos,receita,ticket_medio
0,beleza_saude,8647,1233131.72,130.28
1,relogios_presentes,5495,1166176.98,199.04
2,cama_mesa_banho,9272,1023434.76,93.44
3,esporte_lazer,7530,954852.55,113.25
4,informatica_acessorios,6530,888724.61,116.26
5,moveis_decoracao,6307,711927.69,87.25
6,utilidades_domesticas,5743,615628.69,90.60
7,cool_stuff,3559,610204.10,164.12
8,automotivo,3810,578966.65,139.85
9,brinquedos,3804,471286.48,116.94


# 5. Prazo médio de entrega por estado:

In [5]:
con.execute("""
    SELECT
        c.customer_state                                              AS estado,
        COUNT(DISTINCT o.order_id)                                    AS total_pedidos,
        ROUND(AVG(DATE_DIFF('day',
            CAST(o.order_purchase_timestamp AS DATE),
            CAST(o.order_delivered_customer_date AS DATE))), 1)       AS prazo_medio_dias
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY estado
    ORDER BY prazo_medio_dias DESC
""").fetchdf()

,estado,total_pedidos,prazo_medio_dias
0,RR,41,29.3
1,AP,67,27.2
2,AM,145,26.4
3,AL,397,24.5
4,PA,946,23.7
5,SE,335,21.5
6,MA,717,21.5
7,CE,1279,21.2
8,AC,80,21.0
9,PB,517,20.4


# 6. Nota média de avaliação por cada estado:

In [6]:
con.execute("""
    SELECT
        c.customer_state            AS estado,
        ROUND(AVG(r.review_score), 2) AS nota_media,
        COUNT(*)                    AS total_avaliacoes
    FROM reviews r
    JOIN orders o    ON r.order_id = o.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY estado
    ORDER BY nota_media ASC
""").fetchdf()

,estado,nota_media,total_avaliacoes
0,RR,3.61,46
1,AL,3.75,414
2,MA,3.76,746
3,SE,3.81,349
4,CE,3.85,1329
5,PA,3.85,968
6,BA,3.86,3357
7,RJ,3.87,12765
8,PI,3.92,491
9,PE,4.01,1646


# 7. Taxa de cancelamentos por cada estado:

In [7]:
con.execute("""
    SELECT
        c.customer_state                                        AS estado,
        COUNT(*)                                                AS total_pedidos,
        SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END) AS cancelados,
        ROUND(100.0 * SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END)
              / COUNT(*), 2)                                    AS taxa_cancelamento_pct
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY estado
    ORDER BY taxa_cancelamento_pct DESC
""").fetchdf()

,estado,total_pedidos,cancelados,taxa_cancelamento_pct
0,RR,46,1.0,2.17
1,RO,253,3.0,1.19
2,PI,495,4.0,0.81
3,SP,41746,327.0,0.78
4,RJ,12852,86.0,0.67
5,GO,2020,13.0,0.64
6,MG,11635,64.0,0.55
7,MA,747,4.0,0.54
8,SC,3637,19.0,0.52
9,CE,1336,7.0,0.52
